# 03 — Frozen downstream evaluation

A representation is useful only if it transfers. This notebook uses exactly the same cached, dated task tables for engineered-feature baselines and frozen account-embedding probes. Train, validation, and test remain account-disjoint. A record can only see transactions and life-long profile events available at its cutoff.

The evaluation order matters: create the task table and baseline first, fit a linear probe on frozen embeddings second, then use validation to decide whether LoRA adaptation is worth attempting. The test split stays untouched until the selected model is loaded once.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = Path('/content/FinancialBertForTransactions')
assert (PROJECT_ROOT / 'pyproject.toml').exists(), 'Clone the repository under /content first.'
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST_ROOT = Path('/content/drive/MyDrive/FinancialBertForTransactions')
else:
    PERSIST_ROOT = PROJECT_ROOT
CHECKPOINT = PERSIST_ROOT / 'checkpoints' / 'pragma_lite_mlm' / 'best.pt'
REPORT_DIR = PERSIST_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
assert CHECKPOINT.exists(), f'Run notebook 02 first or set CHECKPOINT: {CHECKPOINT}'

## Current interpretation before fine-tuning

The loan-trouble diagnostic is not a fine-tuning target: only eight positive held-out loans were available, and engineered tabular features beat the frozen embedding on that small test. That is a useful negative result, not a reason to tune harder.

For cash-flow stress, the tabular logistic baseline had a held-out ROC-AUC around 0.82 and average precision around 0.48; the frozen Transformer probe was close, around 0.81 ROC-AUC and 0.46 AP. For the 180-day future transaction-volume proxy, the frozen Ridge probe achieved a lower held-out MAE (about 0.261) than tabular Ridge (about 0.275). That slight edge makes future value the primary LoRA experiment. These are exploratory results, not true LTV or production-risk claims.

In [ ]:
def run_forward_task(task: str) -> tuple[dict, dict]:
    baseline = REPORT_DIR / f'{task}_tabular_baseline.json'
    task_table = REPORT_DIR / f'{task}_task_table.parquet'
    frozen = REPORT_DIR / f'{task}_frozen_probe.json'
    subprocess.run([sys.executable, 'scripts/run_forward_task_baseline.py', '--task', task, '--output', str(baseline)], cwd=PROJECT_ROOT, check=True)
    subprocess.run([sys.executable, 'scripts/run_forward_embedding_probe.py', '--task', task, '--checkpoint', str(CHECKPOINT), '--task-table', str(task_table), '--output', str(frozen), '--device', 'cuda' if IN_COLAB else 'cpu'], cwd=PROJECT_ROOT, check=True)
    return json.loads(baseline.read_text()), json.loads(frozen.read_text())

cashflow_baseline, cashflow_frozen = run_forward_task('cashflow_stress')
future_value_baseline, future_value_frozen = run_forward_task('future_value')

Each JSON report carries the tabular or frozen-model metrics plus account-clustered bootstrap intervals. The bootstrap resamples accounts—not individual snapshots—so repeat cutoffs for a single account do not overstate certainty. The next notebooks reuse these cached tables rather than recreating labels with different cutoffs.